# GenLab ‚Äî Benchmark de Descarga: HuggingFace vs Google Drive
Mide velocidades de descarga desde HF, subida a Drive, y descarga desde Drive.

**Requisitos:**
1. Runtime ‚Üí T4 GPU (o al menos CPU)
2. Espacio libre en Drive suficiente para el modelo (~8.2 GB para DreamShaper)
3. HF_TOKEN configurado (solo para modelos con gated access)

In [ ]:
# 1. Instalar dependencias
!pip install -q huggingface-hub psutil pyyaml tqdm

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BENCH_DIR = '/content/drive/MyDrive/genlab_benchmark'
!mkdir -p "$DRIVE_BENCH_DIR"

In [ ]:
# 3. Clonar repo y cargar genlab
import os, sys, time, shutil, json
from pathlib import Path
from tqdm.notebook import tqdm

REPO_URL = 'https://github.com/ke1npro/GenLab.git'
GENLAB_SRC = '/content/genlab'

if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

from genlab import GenLab
from genlab.models.registry import detect_model
from genlab.config.loader import load_config
from huggingface_hub import snapshot_download, hf_hub_url
import requests

print('genlab importado OK')

In [ ]:
# 4. Elegir modelo para benchmark
DEFAULT_MODEL = "Lykon/dreamshaper-7"
DEFAULT_SIZE_GB = 8

print(f"Modelo por defecto: {DEFAULT_MODEL} (~{DEFAULT_SIZE_GB} GB)")
custom = input("Model ID custom (Enter para usar default): ").strip()
model_id = custom or DEFAULT_MODEL
print(f"\nModelo seleccionado: {model_id}")

In [ ]:
# 5. Limpiar cach√© local del modelo
CACHE_DIR = f"/content/models_cache/{model_id.replace('/', '__')}"
if os.path.isdir(CACHE_DIR):
    print(f"Limpiando cach√© existente: {CACHE_DIR}")
    shutil.rmtree(CACHE_DIR)
    print("[OK] Cach√© eliminada.")
else:
    print("[OK] No hay cach√© previa.")

DRIVE_MODEL_DIR = f"{DRIVE_BENCH_DIR}/{model_id.replace('/', '__')}"
if os.path.isdir(DRIVE_MODEL_DIR):
    shutil.rmtree(DRIVE_MODEL_DIR)
    print("[OK] Cach√© en Drive eliminada.")

In [ ]:
# 6. Benchmark: Descarga desde HuggingFace
def format_bytes(b):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if b < 1024:
            return f'{b:.1f} {unit}'
        b /= 1024
    return f'{b:.1f} TB'

def measure_download(src_label, download_fn, *args, **kwargs):
    """Ejecuta download_fn, mide tiempo y velocidad."""
    start = time.time()
    result = download_fn(*args, **kwargs)
    elapsed = time.time() - start

    # Calcular tama√±o total descargado
    dest = result if isinstance(result, str) else args[0]
    total_bytes = 0
    if os.path.isdir(dest):
        for root, dirs, files in os.walk(dest):
            for f in files:
                fp = os.path.join(root, f)
                if os.path.isfile(fp):
                    total_bytes += os.path.getsize(fp)
    elif os.path.isfile(dest):
        total_bytes = os.path.getsize(dest)

    speed_mbps = (total_bytes / (1024 * 1024)) / elapsed if elapsed > 0 else 0
    return {
        "source": src_label,
        "size": format_bytes(total_bytes),
        "size_bytes": total_bytes,
        "time_s": round(elapsed, 2),
        "speed_mbps": round(speed_mbps, 2),
        "path": dest,
    }

DL_DIR = f'/content/models_cache/{model_id.replace("/", "__")}'
os.makedirs(DL_DIR, exist_ok=True)

print(f"Descargando {model_id} desde HuggingFace...")
hf_result = measure_download(
    "HuggingFace",
    snapshot_download,
    repo_id=model_id,
    local_dir=DL_DIR,
    max_workers=6,
)

print(f"  Tama√±o: {hf_result['size']}")
print(f"  Tiempo: {hf_result['time_s']} s")
print(f"  Velocidad: {hf_result['speed_mbps']} MB/s")
print()

# Guardar resultado
results = [hf_result]

In [ ]:
# 7. Benchmark: Subida a Google Drive
import subprocess

print(f"Subiendo {model_id} a Google Drive...")
print(f"  Origen: {DL_DIR}")
print(f"  Destino: {DRIVE_MODEL_DIR}")

start = time.time()
shutil.copytree(DL_DIR, DRIVE_MODEL_DIR, symlinks=False)
elapsed = time.time() - start

# Calcular tama√±o
total_bytes = 0
for root, dirs, files in os.walk(DRIVE_MODEL_DIR):
    for f in files:
        fp = os.path.join(root, f)
        if os.path.isfile(fp):
            total_bytes += os.path.getsize(fp)

speed_mbps = (total_bytes / (1024 * 1024)) / elapsed if elapsed > 0 else 0
upload_result = {
    "source": "Google Drive (subida)",
    "size": format_bytes(total_bytes),
    "size_bytes": total_bytes,
    "time_s": round(elapsed, 2),
    "speed_mbps": round(speed_mbps, 2),
}

print(f"  Tama√±o: {upload_result['size']}")
print(f"  Tiempo: {upload_result['time_s']} s")
print(f"  Velocidad: {upload_result['speed_mbps']} MB/s")
print()

results.append(upload_result)

In [ ]:
# 8. Benchmark: Descarga desde Google Drive
def download_from_drive(src, dst, run=1):
    """Copia desde Drive a local y mide velocidad."""
    if os.path.isdir(dst):
        shutil.rmtree(dst)

    start = time.time()
    shutil.copytree(src, dst, symlinks=False)
    elapsed = time.time() - start

    total_bytes = 0
    for root, dirs, files in os.walk(dst):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.isfile(fp):
                total_bytes += os.path.getsize(fp)

    speed_mbps = (total_bytes / (1024 * 1024)) / elapsed if elapsed > 0 else 0
    return {
        "source": f"Google Drive (descarga #{run})",
        "size": format_bytes(total_bytes),
        "size_bytes": total_bytes,
        "time_s": round(elapsed, 2),
        "speed_mbps": round(speed_mbps, 2),
    }

DRIVE_DL_DIR = f'/content/models_cache_drive/{model_id.replace("/", "__")}'
os.makedirs('/content/models_cache_drive', exist_ok=True)

# Primera descarga desde Drive
print(f"Descargando {model_id} desde Google Drive (paso 1)...")
drive_result = download_from_drive(DRIVE_MODEL_DIR, DRIVE_DL_DIR, run=1)
print(f"  Tama√±o: {drive_result['size']}")
print(f"  Tiempo: {drive_result['time_s']} s")
print(f"  Velocidad: {drive_result['speed_mbps']} MB/s")
print()
results.append(drive_result)

# Si velocidad > 75 MB/s, repetir 2 veces m√°s para promediar
if drive_result['speed_mbps'] > 75:
    num_extra = 2
    print(f"Velocidad > 75 MB/s ({drive_result['speed_mbps']} MB/s) ó repitiendo {num_extra} veces m\u00e1s...")
    for i in range(2, num_extra + 2):
        print(f"Descarga desde Drive (paso {i})...")
        r = download_from_drive(DRIVE_MODEL_DIR, DRIVE_DL_DIR, run=i)
        print(f"  Velocidad: {r['speed_mbps']} MB/s")
        results.append(r)
        # Peque√±a pausa entre intentos
        time.sleep(2)
else:
    print(f"Velocidad <= 75 MB/s ({drive_result['speed_mbps']} MB/s) ó no se repite.")

# Limpiar descarga de Drive
if os.path.isdir(DRIVE_DL_DIR):
    shutil.rmtree(DRIVE_DL_DIR)

In [ ]:
# 9. Resultados
print("=" * 65)
print(f"{'Origen':<30} {'Tama√±o':<12} {'Tiempo':<12} {'Velocidad':<12}")
print("-" * 65)
for r in results:
    src = r['source']
    sz = r['size']
    t = f"{r['time_s']} s"
    sp = f"{r['speed_mbps']} MB/s"
    print(f"{src:<30} {sz:<12} {t:<12} {sp:<12}")
print("=" * 65)

# Velocidad promedio de Drive (descarga)
drive_dls = [r for r in results if 'descarga' in r['source']]
if drive_dls:
    avg_speed = sum(r['speed_mbps'] for r in drive_dls) / len(drive_dls)
    print(f"\nVelocidad promedio descarga Drive: {avg_speed:.2f} MB/s")

if hf_result['speed_mbps'] > 0:
    ratio = drive_dls[0]['speed_mbps'] / hf_result['speed_mbps'] if hf_result['speed_mbps'] > 0 else 0
    print(f"Drive es {ratio:.1f}x m\u00e1s r\u00e1pido que HF" if ratio > 1 else f"HF es {1/ratio:.1f}x m\u00e1s r\u00e1pido que Drive" if ratio > 0 else "")

# Guardar resultados a JSON
report_path = f"{DRIVE_BENCH_DIR}/benchmark_{model_id.replace('/', '_')}.json"
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nReporte guardado en: {report_path}")

In [ ]:
# 10. Limpieza final
# Preguntar si mantener la descarga de HF en cach√©
keep = input("\u00bfMantener el modelo en cach\u00e9 local? (s/N): ").strip().lower()
if keep != 's':
    if os.path.isdir(DL_DIR):
        shutil.rmtree(DL_DIR)
        print("Cach\u00e9 local eliminada.")
else:
    print("Cach\u00e9 local mantenida.")

print("\nBenchmark completo.")